# Faza 1: Raziskovalna analiza podatkov in predprocesiranje

**Zastavljen cilj**: Razumeti in vizualizirati statistične lastnosti podatkov ter v celoti pripraviti očiščeno, transformirano in skalirano numerično bazo brez manjkajočih vrednosti. Ta faza je ključna za uspešno uporabo nevronskih mrež in algoritma XGBoost v nadaljevanju.

Pomembno opozorilo: Celotno predprocesiranje (imputacija in skaliranje) smo "naučili" (*fit*) izključno na učni (train) množici. Naučene transformacije smo nato aplicirali (*transform*) na učno in na testno množico, s čimer smo strogo preprečili uhajanje informacij (ang. *Data Leakage*).

## 1. Uvoz knjižnic in nalaganje podatkov

Najprej smo uvozili orodja za vizualizacijo (`matplotlib`, `seaborn`) in knjižnice za strojno učenje. Naložili smo učno in testno množico iz prejšnje faze.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# TODO (ekipa): Naložite `../data/train_raw.csv` v obliko `train_df` in `../data/test_raw.csv` v `test_df`.
# TODO (ekipa): Za namene raziskovalne analize (EDA) bomo v prvem delu uporabljali samo `train_df`!

## 2. Raziskovalna analiza podatkov (EDA)
Raziskovalno analizo smo izvedli zgolj na naboru za učenje, da ne bi pristransko vplivali na naše odločitve.

### 2.1 Manjkajoče vrednosti in porazdelitev ciljne spremenljivke
Preverili smo količino in strukturo manjkajočih podatkov ter osnovno porazdelitev razredov pri naši ciljni spremenljivki `loan_status`.

In [ ]:
# TODO (ekipa): Prikažite graf manjkajočih vrednosti (npr. s seaborn heatmap nad vrsticami z isnull() ali bar chart z manjkajočimi odstotki na stolpec).
# TODO (ekipa): Narišite bar chart za ciljno spremenljivko `loan_status`. Prikažite in zapišite odstotek razreda 1 (stopnja neplačil oz. default rate).

### 2.2 Porazdelitev ključnih značilk glede na neplačilo
V nadaljevanju dmo raziskali, kako se ključne značilnosti posojil in posojilojemalcev razlikujejo, glede na to, ali je bilo posojilo poplačano ali ne. 
Uporabili smo prekrivajoče se histograme in škatle z brki (boxplots) za prikazane atribute.

In [ ]:
# TODO (ekipa): Narišite histograme/KDE plots z ločenimi barvami (hue='loan_status') za `annual_inc`, `dti`, `int_rate`, `fico_range_low` in `revol_util`.
# TODO (ekipa): Komentirajte ugotovitve (npr. 'Višja obrestna mera je očitno prisotna pri neplačnikih').
# TODO (ekipa): Narišite stolpične diagrame deleža neplačil (default rate) glede na kategorične značilke `grade`, `home_ownership` in `purpose`.

### 2.3 Korelacije in prisotnost besedilnih polj
Raziskali smo medsebojne linearne korelacije med numeričnimi vrednostmi ter preverili, koliko podatkov vsebuje prave opise posojil (stolpca `desc` in `emp_title`).

In [ ]:
# TODO (ekipa): Narišite korelacijsko matriko (seaborn heatmap) za vse numerične atribute.
# TODO (ekipa): Izračunajte odstotek vrstic, ki IMAJO izpolnjeno polje `desc`, in tistih, ki imajo `emp_title`. 
# (Kot pojasnjeno v načrtu, `desc` pričakujemo okrog ~5 %, `emp_title` pa zelo visoko).

## 3. Predprocesiranje podatkov (Imputacija manjkajočih vrednosti)

Obdelavo smo zastavili skrbno: pravila smo definirali in statistike izračunali na učni množici, nato pa enaka pravila aplicirali še na testno.
1. **Numerične spremenljivke**: Vrednosti z manjšim deležem manjkajočih smo nadomestili z mediano. Odstranili smo morebitne stolpce, ki bi imeli >50% manjajočih vrednosti.
2. **Čas in meseci (`mths_since_last_delinq`)**: Odsotnost vrednosti tu pomeni, da dogodka ni bilo. Teh nismo kar zavrgli; ustvarili smo binarno zastavico `had_delinquency` in nato vrednosti nadomestili z mediano.
3. **Besedilo**: NaN v `desc` smo napolnili s praznim nizom (`""`), v `emp_title` pa z oznako `"unknown"`.

In [ ]:
# TODO (ekipa): Implementirajte imputacijo nad train in apply nad test.
# - Numerični imputator (SimpleImputer s strategijo 'median').
# - Izdelava zastavice `train_df['had_delinquency'] = train_df['mths_since_last_delinq'].notnull().astype(int)` itd.
# - Zamenjava tekstovnih NaN: `desc` -> "" in `emp_title` -> "unknown".

## 4. Inženiring značilk (Feature Engineering)

Z namenom boljšega razumevanja podatkov pri modelih (predvsem LR in XGBoost) smo izpeljali vrsto novih naprednih značilk:
- Povezali smo spodnjo in zgornjo FICO mejo v `fico_avg` ter izbrisali originala.
- Datumska polja (kot `earliest_cr_line`) smo pretvorili v numerično vrednost: `credit_history_years` (čas od nastanka do referenčnega datuma recimo leta 2018).
- Ustvarili smo razmerja: `loan_to_income` (višina kredita deljena z letnim prihodkom) in `installment_to_income` (obrok napram mesečnemu prihodku).

In [ ]:
# TODO (ekipa): Izračunaj `fico_avg = (fico_range_low + fico_range_high) / 2` in odvrzi prejšnja stolpca (tako za train kot test!).
# TODO (ekipa): Izračunaj `credit_history_years` iz `earliest_cr_line`. Parsiraj string oblik 'Dec-1998' in ugotovi starost. Odvrzi `earliest_cr_line`.
# TODO (ekipa): Ustvari stolpec `loan_to_income` in `installment_to_income`. Pazite na deljenje z 0 (dodajte mikroskopski epsilon k delitelju).

## 5. Kodiranje kategoričnih spremenljivk

Da se algoritmi uspešno učijo, smo strojne besede morali preslikati v numerične vrednosti:
- **Ordinalne spremenljivke**: `grade` (A=1...G=7), `sub_grade` (A1=1...G5=35), `term` (36 ali 60 v celo število).
- **Dolžina zaposlitve (`emp_length`)**: pretvorili smo v enostavne številke (0 do 10 let), manjkajoče pripisali stiski/mediani.
- **Kategorične spremenljivke (`home_ownership`, `purpose`, `application_type`, `verification_status`)**: uporabili smo nominalno vroče kodiranje (ang. *One-Hot Encoding*) s spustitvijo prvega stolpca (da preprečimo kolinearnost).

In [ ]:
# TODO (ekipa): Napišite ročno mapiranje za `grade` (A=1, B=2...), `sub_grade` in odstranite " months" v `term`.
# TODO (ekipa): Ustrezno ekstrahirajte številko iz `emp_length` (npr. '< 1 year' -> 0, '10+ years' -> 10).
# TODO (ekipa): Izvedi One-Hot encoding z pd.get_dummies(..., drop_first=True) za nominalne kategorije. Pazi na ujemanje stolpcev v train in test!

## 6. Skaliranje

Zaključni korak manipulacije pred modeliranjem je predstavljalo skaliranje numeričnih značilk, saj logistična regresija (in do neke mere K-means) podleže napačnemu vplivu absolutne velikosti enot (npr. dohodki v tisočih napram starosti polja v letih).
Aplicirali smo `StandardScaler` in ga prilagodili striktno na **učni** množici, nato smo transformirali še testno.

In [ ]:
# TODO (ekipa): Initializiraj StandardScaler, izvedi .fit() na `train_df` in ustrezno  .transform() na obeh množicah.
# TODO (ekipa): Skalirati ne smete One-Hot enkodiranih stolpcev, tekstovnih ali pa same ciljne spremenljivke (loan_status). Bodite previdni.

## 7. Izvoz obdelanih podatkov

Pripravljene numerične in besedilne atribute smo shranili kot nove `.csv` datoteke, objekte skaliranja operacij pa previdno shranili s knjižnico `joblib` v projektno mapo z modeli, da bi jih lahko ob interaktivni oceni prihodnjih obiskovalcev preko aplikacije vedno naložili nazaj v isti obliki.

In [ ]:
# TODO (ekipa): Preverite in vizualizirajte za občasne NaN napake ter izpišite dimenzije dokončanih množic. Pazite da niste izgubili stolpcev `desc` in `emp_title`.
# TODO (ekipa): Shranite procesirane Dataframe-e: `train_processed.csv` in `test_processed.csv` v preklopni imenik `../data/`.
# TODO (ekipa): Obvezno izvozite scaler z `joblib.dump(scaler, '../models/scaler.pkl')`.